# 03 — Diagnostic table + classifier validation
Part 6's deliverable-in-its-own-right (the per-category failure profile), plus Part 4's classifier-agreement check, which belongs in the report regardless of what the curriculum result looks like.

In [ ]:
%run notebooks/00_setup.ipynb

## Validate the classifier BEFORE trusting the table (Part 4)
Target agreement >= 85%. Do this before building the curriculum reweighting on top of it -- put the number in the report either way.

In [ ]:
!python -m scripts.build_eval_verify_results \
  --adapter artifacts/m0/final \
  --eval-sets data/eval/verilogeval_v2.jsonl data/eval/rtllm_v2.jsonl \
  --n 3 --out artifacts/eval_verify_results.jsonl

In [ ]:
# Requires a verify-results jsonl with per-record stage/ok/error_label --
# generate one quickly against the eval sets (real testbenches) rather than
# the testbench-less probe split, since hand-labelling needs ground truth
# you can actually check against.
!python -m scripts.validate_classifier sample \
  --results artifacts/eval_verify_results.jsonl --n 100 \
  --out artifacts/hand_label_sample.jsonl
print('Now open artifacts/hand_label_sample.jsonl and fill in human_label for each row.')

In [ ]:
# After hand-labelling:
!python -m scripts.validate_classifier score --labeled artifacts/hand_label_sample.jsonl

## Taxonomy table (Part 6)

In [ ]:
import json
import pandas as pd
diag = json.load(open('artifacts/m0_diagnostic.json'))
df = pd.DataFrame(diag['labels'])
df['share_of_failures'] = (df['share_of_failures'] * 100).round(1)
print(f"catch-all share (wrong_logic_other + *_other): {diag['catch_all_share']:.1%}")
df

In [ ]:
# Tier-concentration heatmap -- "where" the curriculum should intervene
import matplotlib.pyplot as plt
tiers = ['T1', 'T2', 'T3', 'T4']
mat = [[row['tier_concentration'].get(t, 0.0) for t in tiers] for row in diag['labels']]
fig, ax = plt.subplots(figsize=(6, max(3, 0.35 * len(diag['labels']))))
im = ax.imshow(mat, aspect='auto', cmap='viridis')
ax.set_xticks(range(len(tiers))); ax.set_xticklabels(tiers)
ax.set_yticks(range(len(diag['labels']))); ax.set_yticklabels([r['label'] for r in diag['labels']])
fig.colorbar(im, label='share of this label\'s failures')
plt.title('Error-label x structural-tier concentration (M0 probe split)')
plt.tight_layout()
plt.savefig('artifacts/tier_concentration_heatmap.png')
plt.show()